# Text Representation
- This is the most important part of the NLP FLow, as machines do not understand any language, so text need to be converted to numbers

In [ ]:
import pandas as pd

In [ ]:
dataset=pd.read_csv('IMDB Dataset.csv', encoding='utf-8', engine='python', on_bad_lines='skip')

In [ ]:
dataset.shape

(31641, 2)

## 1. OHE- The simplest way to convert text into numbers

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical


tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset['review'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Total vocabulary size (V): {vocab_size}")

review_sequences = tokenizer.texts_to_sequences(dataset['review'])

Total vocabulary size (V): 100953


In [ ]:
def get_one_hot_sentence_representation(sequence, vocab_size):

    if not sequence:
        return np.empty((0, vocab_size))

    n = len(sequence)

    one_hot_matrix = to_categorical(sequence, num_classes=vocab_size)
    return one_hot_matrix

In [ ]:
first_review_sequence = review_sequences[0]

first_review_ohe_matrix = get_one_hot_sentence_representation(first_review_sequence, vocab_size)
print(f"Shape of the first review's (n, V) representation: {first_review_ohe_matrix.shape}")

second_review_sequence = review_sequences[1]

second_review_ohe_matrix = get_one_hot_sentence_representation(second_review_sequence, vocab_size)
print(f"Shape of the second review's (n, V) representation: {second_review_ohe_matrix.shape}")

Shape of the first review's (n, V) representation: (314, 100953)
Shape of the second review's (n, V) representation: (164, 100953)


In [ ]:
print("First review's one-hot encoded matrix:")
display(first_review_ohe_matrix)

print("Second review's one-hot encoded matrix:")
display(second_review_ohe_matrix)

First review's one-hot encoded matrix:


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

Second review's one-hot encoded matrix:


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Sample sentences
sentences = [
    "the cat sat",
    "dog chased cat"
]

# 2. Tokenize text (maps each unique word to a unique integer index)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

# Find vocabulary size (plus 1 for padding/0-index token)
vocab_size = len(tokenizer.word_index) + 1

print("Word to Index Mapping:", tokenizer.word_index)
print("Sentences mapped to Integer Sequences:", sequences)

# 3. Convert integer sequences into a 3D One-Hot Encoded Tensor
# Shape: (Number of sentences, Number of words in sentence, Vocabulary Size)
word_level_ohe = [to_categorical(seq, num_classes=vocab_size) for seq in sequences]

print("\nWord-level OHE Vector for the first word ('the') in the first sentence:")
print(word_level_ohe[0][0])  # Prints the OHE array for 'the'


Word to Index Mapping: {'cat': 1, 'the': 2, 'sat': 3, 'dog': 4, 'chased': 5}
Sentences mapped to Integer Sequences: [[2, 1, 3], [4, 5, 1]]

Word-level OHE Vector for the first word ('the') in the first sentence:
[0. 0. 1. 0. 0. 0.]


##
- Shape of the first review's (n, V) representation: (314, 124253)
Shape of the second review's (n, V) representation: (164, 124253)
- form this output we can conclude that each input text is not of the same length making it ununiform for the ML models which expect the uniform length input
- OOV words: Test data can have out of vocab words
- the data we have is bery sparse
- Schemantic meaning of the data is not captured

## 2. Bag-of-Words
- Core idea is to represent text by looking at word frequencies while completely ignoring word order, grammar, and sentence structure

In [ ]:
df = pd.DataFrame({'text':['people watch campusx','campusx watch campusx','people write comment','campusx write comment'],'output':[1,1,0,0]})


In [ ]:
df

,text,output
0,people watch campusx,1
1,campusx watch campusx,1
2,people write comment,0
3,campusx write comment,0


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

In [ ]:
bow=cv.fit_transform(df['text'])

In [ ]:
print(cv.vocabulary_)

{'people': 2, 'watch': 3, 'campusx': 0, 'write': 4, 'comment': 1}


In [ ]:
bow[0].toarray()

array([[1, 0, 1, 1, 0]])

In [ ]:
cv.transform(["campusx watch and write comment of campusx"]).toarray()

array([[2, 1, 0, 1, 1]])

In [ ]:
# Issues Resolved-
# 1. Out of vocab- here ex- and is not not in vocab so it will not be counted
# 1. Fiext length of each vector each will be same as Vocab size

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer() # Initialize CountVectorizer
bow_matrix = cv.fit_transform(dataset['review'])

print(f"Shape of the Bag-of-Words matrix: {bow_matrix.shape}")
print("Vocabulary size:", len(cv.vocabulary_))
print("First 5 vocabulary items:", list(cv.vocabulary_.items())[:5])

# Displaying the first review's Bag-of-Words vector (as a dense array for readability)
print("\nFirst review's Bag-of-Words vector (first 50 dimensions):\n", bow_matrix[0].toarray()[0][:])

Shape of the Bag-of-Words matrix: (31641, 84387)
Vocabulary size: 84387
First 5 vocabulary items: [('one', 52980), ('of', 52690), ('the', 74728), ('other', 53509), ('reviewers', 62571)]

First review's Bag-of-Words vector (first 50 dimensions):
 [0 0 0 ... 0 0 0]


## 3. N-gram

In [ ]:
# bi-gram
cv2=CountVectorizer(ngram_range=(2, 2))

In [ ]:
bow=cv2.fit_transform(df['text'])

In [ ]:
print(cv2.vocabulary_)

{'people watch': 2, 'watch campusx': 4, 'campusx watch': 0, 'people write': 3, 'write comment': 5, 'campusx write': 1}


In [ ]:
bow[0].toarray()

array([[0, 0, 1, 0, 1, 0]])

In [ ]:
cv2.transform(["campusx watch and write comment of campusx"]).toarray()

array([[1, 0, 0, 0, 0, 1]])

In [ ]:
# uni+bi-gram
cv3=CountVectorizer(ngram_range=(1, 2))

In [ ]:
bow=cv3.fit_transform(df['text'])

In [ ]:
print(cv3.vocabulary_)

{'people': 4, 'watch': 7, 'campusx': 0, 'people watch': 5, 'watch campusx': 8, 'campusx watch': 1, 'write': 9, 'comment': 3, 'people write': 6, 'write comment': 10, 'campusx write': 2}


In [ ]:
bow[0].toarray()

array([[1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0]])

In [ ]:
cv3.transform(["campusx watch and write comment of campusx"]).toarray()

array([[2, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1]])

## 5. Tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()

In [ ]:
tfidf.fit_transform(df['text']).toarray()
# tf*idf

array([[0.49681612, 0.        , 0.61366674, 0.61366674, 0.        ],
       [0.8508161 , 0.        , 0.        , 0.52546357, 0.        ],
       [0.        , 0.57735027, 0.57735027, 0.        , 0.57735027],
       [0.49681612, 0.61366674, 0.        , 0.        , 0.61366674]])

In [ ]:
print(tfidf.idf_)
print(tfidf.get_feature_names_out())

[1.22314355 1.51082562 1.51082562 1.51082562 1.51082562]
['campusx' 'comment' 'people' 'watch' 'write']


In [ ]:
tfidf.fit_transform(dataset['review']).toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
print(tfidf.idf_)
print(tfidf.get_feature_names_out())

[ 6.72751164  5.72033356 10.66909345 ... 10.66909345 10.66909345
 10.66909345]
['00' '000' '00000000000' ... 'üzümcü' 'þorleifsson' 'żmijewski']
